# Lung Sounds, End to End
### IEEE BSN Workshop · hands-on session

You will take real chest recordings through a full classification pipeline:
**listen → filter → segment → mel spectrogram → train a classifier → export to ONNX**.
Every step is controlled with menus and sliders, so no coding is needed… unless you want to try! :)

**Dataset.** ICBHI 2017: chest recordings in which every breathing cycle is annotated for
*crackles* and *wheezes*. That gives four classes: **Normal**, **Crackle**, **Wheeze** and **Both**.

**How to use it.** Run the cells from top to bottom (or *Runtime → Run all*), then play with the
controls under each section.

---
## 1 | Setup

Installs anything missing and downloads the data (sample recordings and precomputed model
embeddings, about 150 MB). Nothing to change here, just run the two cells.

In [ ]:
# --- environment -------------------------------------------------------------
import importlib, subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules

def ensure(pkg, import_name=None):
    """Install pkg only if it isn't importable. Keeps Colab startup quiet."""
    try:
        importlib.import_module(import_name or pkg)
        return True
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
        try:
            importlib.import_module(import_name or pkg)
            return True
        except ImportError:
            print(f"  could not import {import_name or pkg} after install")
            return False

for pkg, mod in [("numpy", "numpy"), ("scipy", "scipy"), ("librosa", "librosa"),
                 ("soundfile", "soundfile"), ("matplotlib", "matplotlib"),
                 ("ipywidgets", "ipywidgets")]:
    ensure(pkg, mod)

if IN_COLAB:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()

import json, glob, math, warnings
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, Audio, clear_output

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "figure.autolayout": True,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 9})
print(f"python {sys.version.split()[0]} | numpy {np.__version__} | colab={IN_COLAB}")

In [ ]:
# --- locate the data ---------------------------------------------------------
# Priority: BSN_DATA_DIR env var  ->  ./bsn_workshop_data  ->  download from Hugging Face.
HF_REPO, HF_FOLDER = "AlirezaFB/ICBHI_2017", "workshop"

HELP = f"""
Could not find the data bundle.

Fix it in one of two ways:

  1. Point at a local copy:
         import os; os.environ["BSN_DATA_DIR"] = r"/path/to/bsn_workshop_data"
     then re-run this cell. The folder must contain manifest.json.

  2. Make sure {HF_REPO} (folder {HF_FOLDER}/) is reachable and public,
     or log in first with:  from huggingface_hub import login; login()
"""

def _looks_like_bundle(d):
    return bool(d) and os.path.isfile(os.path.join(d, "manifest.json"))

def find_data_dir():
    # 1. explicit override
    if _looks_like_bundle(os.environ.get("BSN_DATA_DIR")):
        return os.environ["BSN_DATA_DIR"]

    # 2. look beside the notebook and a couple of levels up
    here = os.path.abspath(os.getcwd())
    for up in range(3):
        base = here if up == 0 else os.path.abspath(os.path.join(here, *([".."] * up)))
        for sub in ("bsn_workshop_data", os.path.join("data", "bsn_workshop_data")):
            c = os.path.join(base, sub)
            if _looks_like_bundle(c):
                return c
    if _looks_like_bundle("/content/bsn_workshop_data"):
        return "/content/bsn_workshop_data"

    # 3. Hugging Face
    if not ensure("huggingface_hub"):
        raise RuntimeError("huggingface_hub is unavailable." + HELP)
    from huggingface_hub import snapshot_download
    print(f"no local bundle found; downloading {HF_REPO}:{HF_FOLDER}/ ...")
    try:
        # audio/ and embeddings/ may sit inside workshop/ or at the repo root
        root = snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                                 allow_patterns=[f"{HF_FOLDER}/*", "audio/*", "embeddings/*"])
    except Exception as e:
        raise RuntimeError(f"Hugging Face download failed ({type(e).__name__}: "
                           f"{str(e).splitlines()[0]})." + HELP) from None
    d = os.path.join(root, HF_FOLDER)
    for sub in ("audio", "embeddings"):           # link root-level folders into the bundle
        if not os.path.isdir(os.path.join(d, sub)) and os.path.isdir(os.path.join(root, sub)):
            os.symlink(os.path.join(root, sub), os.path.join(d, sub))
    if not _looks_like_bundle(d):
        raise RuntimeError(f"Downloaded {HF_REPO} but {HF_FOLDER}/manifest.json is missing." + HELP)
    return d

DATA_DIR = find_data_dir()
MANIFEST = json.load(open(os.path.join(DATA_DIR, "manifest.json"), encoding="utf-8"))

# Class order comes from the manifest. Never hard-code it.
LABELS   = [MANIFEST["label_map"][str(i)] for i in range(len(MANIFEST["label_map"]))]
LABEL_IX = {name: i for i, name in enumerate(LABELS)}
NORMAL   = LABEL_IX["Normal"]
ABNORMAL = [i for i, n in enumerate(LABELS) if n != "Normal"]

# Defaults for the DSP sections, also from the manifest.
_pp  = MANIFEST["preprocessing"]
SR_D = int(_pp["resample_rate_hz"])
LO_D, HI_D = float(_pp["filter"]["lowcut_hz"]), float(_pp["filter"]["highcut_hz"])
ORD_D      = int(_pp["filter"]["order"])

# The sample recordings used in sections 2-5.
RECS    = {r["recording"]: r for r in MANIFEST["sample_recordings"]["recordings"]}
DEVICES = sorted({r["device"] for r in RECS.values()})

print(f"Data ready: {len(RECS)} sample recordings from {len(DEVICES)} devices, "
      f"embeddings for {len(MANIFEST['models'])} models.")
print(f"Classes: {', '.join(LABELS)}")

---
## 2 | Listen to the data

Pick a device and a recording to see its waveform, with every annotated breathing cycle coloured by
its class, and play it.

**Try:** tick *hide normal cycles* to see where the abnormal sounds are, then listen for them.
Crackles are short clicks; wheezes are longer, musical tones. Compare devices too: each stethoscope
sounds different.

In [ ]:
import soundfile as sf
import librosa
from scipy.signal import butter, sosfiltfilt, sosfilt

PLAY_SR = 8000          # cap playback rate: keeps the notebook light
CLR = {"Normal": "#9aa5ad", "Wheeze": "#0b6e75", "Crackle": "#c2622a", "Both": "#8e3b8e"}

def load_recording(name):
    """Return (signal float64 mono, sample_rate). No resampling."""
    y, sr = sf.read(os.path.join(DATA_DIR, "audio", f"{name}.wav"), dtype="float64")
    if y.ndim > 1:
        y = y.mean(axis=1)
    return y, sr

def load_cycles(name):
    """Read the ICBHI annotation (start, end, crackle, wheeze) -> list of (start_s, end_s, class)."""
    out = []
    with open(os.path.join(DATA_DIR, "audio", f"{name}.txt")) as fh:
        for line in fh:
            p = line.split()
            if len(p) != 4:
                continue
            st, en, cr, wh = float(p[0]), float(p[1]), int(float(p[2])), int(float(p[3]))
            cr, wh = bool(cr), bool(wh)
            name_of = "Both" if (cr and wh) else "Crackle" if cr else "Wheeze" if wh else "Normal"
            out.append((st, en, LABEL_IX[name_of]))
    return out

def for_playback(y, sr, target=PLAY_SR):
    """Downsample for the audio widget only. Never used for analysis."""
    if sr <= target:
        return y, sr
    return librosa.resample(y, orig_sr=sr, target_sr=target), target

def show_recording(name, hide_normal=False):
    y, sr = load_recording(name)
    cyc   = load_cycles(name)
    t     = np.arange(len(y)) / sr
    fig, ax = plt.subplots(figsize=(11, 2.6))
    ax.plot(t, y, lw=0.4, color="#2b3036", zorder=3)
    seen = set()
    for st, en, k in cyc:
        nm = LABELS[k]
        if hide_normal and k == NORMAL:
            continue
        ax.axvspan(st, en, color=CLR[nm], alpha=0.30, lw=0, zorder=1,
                   label=nm if nm not in seen else None)
        seen.add(nm)
    ax.set_xlim(0, t[-1]); ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
    ax.set_title(f"{name}   |   {RECS[name]['device']}   |   {sr} Hz   |   "
                 f"{len(cyc)} cycles" + ("   (normal cycles hidden)" if hide_normal else ""))
    if seen:
        ax.legend(loc="upper right", ncol=len(seen), fontsize=8, framealpha=0.9)
    plt.show()
    yp, sp = for_playback(y, sr)
    display(Audio(yp, rate=sp))

In [ ]:
# --- interactive ---
_w_dev  = W.Dropdown(options=DEVICES, value=DEVICES[0], description="Device:")
_w_rec  = W.Dropdown(description="Recording:", layout=W.Layout(width="420px"))
_w_hide = W.Checkbox(value=False, description="hide normal cycles", indent=False)
_out2   = W.Output()

def _recs_for(dev):
    return [n for n, r in RECS.items() if r["device"] == dev]

def _refresh2(*_):
    with _out2:
        clear_output(wait=True)
        if _w_rec.value:
            show_recording(_w_rec.value, _w_hide.value)

def _on_dev(*_):
    _w_rec.options = _recs_for(_w_dev.value)
    _w_rec.value = _w_rec.options[0]

_w_dev.observe(_on_dev, "value")
_w_rec.observe(_refresh2, "value")
_w_hide.observe(_refresh2, "value")
_on_dev()
display(W.VBox([W.HBox([_w_dev, _w_rec]), _w_hide, _out2]))
_refresh2()

---
## 3 | Filtering

Resamples a recording and applies a Butterworth filter. You see the waveform and the spectrum before
and after, and can listen to both.

| Setting | What it does |
|---|---|
| **Filter** | *low-pass* keeps frequencies below the high cut, *high-pass* keeps those above the low cut, *band-pass* keeps the band in between, *none* switches filtering off. |
| **low cut** | Removes what lies below it. Heart sounds and body noise sit mostly below ~150 Hz. |
| **high cut** | Removes what lies above it, such as hiss and friction noise. Lung sounds carry most of their energy below ~2000 Hz. |
| **order** | How steep the cut is. Higher means a sharper edge but more ringing. |
| **sample rate** | The rate the audio is resampled to. The highest frequency it can hold is half of it (2000 Hz at 4 kHz). |

The defaults (band-pass 80–1900 Hz, order 5, 4 kHz) are the settings used to make the model
embeddings in section 6.

**Try:** raise the low cut to 300 Hz and hear the heartbeat disappear; lower the high cut to 500 Hz
and hear the crackles lose their sharpness.

In [ ]:
NYQ_MARGIN = 0.98   # keep cutoffs strictly below Nyquist

def apply_filter(y, sr, kind="band", lo=LO_D, hi=HI_D, order=ORD_D):
    """Butterworth filter, zero-phase. kind: none | low | high | band."""
    if kind == "none":
        return y.copy()
    nyq = 0.5 * sr
    lo_n, hi_n = lo / nyq, min(hi / nyq, NYQ_MARGIN)
    if kind == "low":
        if hi_n <= 0: return y.copy()
        sos = butter(order, hi_n, btype="low", output="sos")
    elif kind == "high":
        if lo_n <= 0: return y.copy()
        sos = butter(order, lo_n, btype="high", output="sos")
    else:
        if not (0 < lo_n < hi_n < 1):
            raise ValueError(f"band-pass needs 0 < {lo:.0f} < {hi:.0f} < {nyq:.0f} Hz")
        sos = butter(order, [lo_n, hi_n], btype="band", output="sos")
    # filtfilt needs a signal longer than the filter's padding
    if len(y) <= 3 * sos.shape[0] * 2:
        return sosfilt(sos, y)
    return sosfiltfilt(sos, y)

def spectrum_db(y, sr, ref=None):
    """Single-sided amplitude spectrum in dB: 20*log10|X|. Returns (freqs, dB, ref)."""
    n = int(2 ** math.ceil(math.log2(max(len(y), 2))))
    X = np.abs(np.fft.rfft(y * np.hanning(len(y)), n=n))
    f = np.fft.rfftfreq(n, 1.0 / sr)
    r = ref if ref is not None else max(X.max(), 1e-12)
    return f, 20 * np.log10(np.maximum(X, 1e-12) / r), r

def show_filter(name, kind, lo, hi, order, target_sr):
    y0, sr0 = load_recording(name)
    y = librosa.resample(y0, orig_sr=sr0, target_sr=target_sr) if sr0 != target_sr else y0.copy()
    sr = target_sr
    try:
        yf = apply_filter(y, sr, kind, lo, hi, order)
    except ValueError as e:
        print(f"invalid settings: {e}"); return
    t = np.arange(len(y)) / sr
    fig, ax = plt.subplots(2, 2, figsize=(11, 4.6))
    ax[0, 0].plot(t, y,  lw=0.4, color="#2b3036"); ax[0, 0].set_title(f"before  ({sr} Hz)")
    ax[0, 1].plot(t, yf, lw=0.4, color="#0b6e75"); ax[0, 1].set_title(f"after  ({kind})")
    for a in ax[0]:
        a.set_xlabel("Time (s)"); a.set_ylabel("Amplitude"); a.set_xlim(0, t[-1])
    lim = max(np.abs(y).max(), np.abs(yf).max()) * 1.05
    ax[0, 0].set_ylim(-lim, lim); ax[0, 1].set_ylim(-lim, lim)

    f1, d1, ref = spectrum_db(y, sr)
    f2, d2, _   = spectrum_db(yf, sr, ref=ref)
    for a, (f, d, c, ttl) in zip(ax[1], [(f1, d1, "#2b3036", "spectrum before"),
                                         (f2, d2, "#0b6e75", "spectrum after")]):
        a.plot(f, d, lw=0.7, color=c)
        a.set_xlim(0, sr / 2); a.set_ylim(-90, 2)
        a.set_xlabel("Frequency (Hz)"); a.set_ylabel("Amplitude (dB re max)")
        a.set_title(ttl)
        if kind in ("high", "band"):
            a.axvline(lo, color="#c2622a", ls="--", lw=0.9)
        if kind in ("low", "band"):
            a.axvline(min(hi, sr / 2 * NYQ_MARGIN), color="#c2622a", ls="--", lw=0.9)
    plt.show()
    yp, sp = for_playback(y, sr);  print("before:"); display(Audio(yp, rate=sp))
    yp, sp = for_playback(yf, sr); print("after:");  display(Audio(yp, rate=sp))

In [ ]:
# --- interactive ---
_f_rec  = W.Dropdown(options=list(RECS), value=list(RECS)[0], description="Recording:",
                     layout=W.Layout(width="420px"))
_f_kind = W.Dropdown(options=[("none", "none"), ("low-pass", "low"),
                              ("high-pass", "high"), ("band-pass", "band")],
                     value="band", description="Filter:")
_f_lo   = W.FloatSlider(value=LO_D, min=10,  max=900,  step=10, description="low cut (Hz):",
                        continuous_update=False, readout_format=".0f", style={"description_width":"110px"})
_f_hi   = W.FloatSlider(value=HI_D, min=200, max=3900, step=50, description="high cut (Hz):",
                        continuous_update=False, readout_format=".0f", style={"description_width":"110px"})
_f_ord  = W.IntSlider(value=ORD_D, min=1, max=10, step=1, description="order:",
                      continuous_update=False)
_f_sr   = W.Dropdown(options=[4000, 8000, 16000], value=SR_D, description="sample rate:",
                     style={"description_width":"110px"})
_out3   = W.Output()

def _refresh3(*_):
    with _out3:
        clear_output(wait=True)
        show_filter(_f_rec.value, _f_kind.value, _f_lo.value, _f_hi.value,
                    _f_ord.value, _f_sr.value)

for _w in (_f_rec, _f_kind, _f_lo, _f_hi, _f_ord, _f_sr):
    _w.observe(_refresh3, "value")
display(W.VBox([W.HBox([_f_rec, _f_kind]), W.HBox([_f_lo, _f_hi]),
                W.HBox([_f_ord, _f_sr]), _out3]))
_refresh3()

### ✏️ Try it yourself: your own filter

Do you know another denoising or filtering method (wavelet denoising, spectral subtraction, a Wiener
filter, ...)? Implement it below, plot its result and compare it with the Butterworth filter above.
You can reuse `load_recording(name)` and `apply_filter(y, sr, kind, lo, hi, order)`.

*The cell below is yours to experiment in.*

In [ ]:
# Your code here

---
## 4 | Segmentation

The models need inputs of a fixed length, but breathing cycles vary in length. Here you cut one
annotated cycle out of a recording (after the default band-pass filter) and force it to a fixed
duration: shorter cycles are padded, longer ones are truncated. The padded part is shaded orange.

| Setting | What it does |
|---|---|
| **Cycle** | Which annotated breathing cycle to cut. Each entry shows its position in the recording, its class and its annotated length. |
| **duration** | The target length. The embeddings used 4 s. |
| **padding** | How a short cycle is filled up: *reflect* mirrors the signal (used for the embeddings), *zeros* adds silence, *repeat* loops the cycle. |
| **sample rate** | The rate the audio is resampled to before cutting. |

**Try:** pick a short cycle, set the duration to 8 s and compare the three padding modes.

In [ ]:
def cut_cycle(y, sr, start_s, end_s, duration_s, mode="reflect"):
    """Cut [start_s, end_s) and force it to duration_s.

    Returns (segment, pad_start_samples, pad_end_samples). Padding is appended at the end,
    matching the pipeline that produced the embeddings.
    """
    a, b = int(start_s * sr), int(end_s * sr)
    seg  = y[max(a, 0):min(b, len(y))]
    want = int(round(duration_s * sr))
    if len(seg) == 0:
        return np.zeros(want), 0, want
    if len(seg) >= want:
        return seg[:want], 0, 0
    need = want - len(seg)
    if mode == "zeros":
        out = np.concatenate([seg, np.zeros(need)])
    elif mode == "repeat":
        reps = int(np.ceil(want / len(seg)))
        out = np.tile(seg, reps)[:want]
    else:  # reflect
        if need >= len(seg):
            reps = int(np.ceil(want / len(seg)))
            out = np.tile(np.concatenate([seg, seg[::-1]]), reps)[:want]
        else:
            out = np.pad(seg, (0, need), mode="reflect")
    return out, 0, need

def cycle_options(name):
    """Menu entries for a recording's cycles, e.g. ('Cycle 3 of 8 · Crackle · 1.8 s', 2)."""
    cyc = load_cycles(name)
    return [(f"Cycle {i + 1} of {len(cyc)} · {LABELS[k]} · {en - st:.1f} s", i)
            for i, (st, en, k) in enumerate(cyc)]

def show_segment(name, cycle_ix, duration_s, mode, target_sr):
    y0, sr0 = load_recording(name)
    y  = librosa.resample(y0, orig_sr=sr0, target_sr=target_sr) if sr0 != target_sr else y0.copy()
    sr = target_sr
    y  = apply_filter(y, sr, "band", LO_D, HI_D, ORD_D)
    cyc = load_cycles(name)
    st, en, k = cyc[cycle_ix]
    seg, _, pad = cut_cycle(y, sr, st, en, duration_s, mode)
    real = len(seg) - pad
    t = np.arange(len(seg)) / sr
    fig, ax = plt.subplots(figsize=(11, 2.4))
    ax.plot(t, seg, lw=0.5, color="#2b3036", zorder=3)
    ax.axvspan(0, real / sr, color=CLR[LABELS[k]], alpha=0.22, lw=0, label="real audio")
    if pad:
        ax.axvspan(real / sr, len(seg) / sr, color="#c2622a", alpha=0.22, lw=0,
                   label=f"{mode} padding ({pad/sr:.2f} s)")
    truncated = (en - st) > duration_s
    ax.set_xlim(0, t[-1]); ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
    ax.set_title(f"Cycle {cycle_ix + 1} of {len(cyc)}  |  {LABELS[k]}  |  "
                 f"annotated {en-st:.2f} s -> {duration_s:.1f} s"
                 + ("   TRUNCATED" if truncated else ""))
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.show()
    pct = 100.0 * pad / len(seg)
    print(f"{real/sr:.2f} s real audio + {pad/sr:.2f} s padding ({pct:.0f}% of the segment is padding)")
    yp, sp = for_playback(seg, sr); display(Audio(yp, rate=sp))

In [ ]:
# --- interactive ---
_s_rec  = W.Dropdown(options=list(RECS), value=list(RECS)[0], description="Recording:",
                     layout=W.Layout(width="420px"))
_s_cyc  = W.Dropdown(description="Cycle:", layout=W.Layout(width="320px"))
_s_dur  = W.FloatSlider(value=4.0, min=0.5, max=10.0, step=0.5, description="duration (s):",
                        continuous_update=False, style={"description_width":"110px"})
_s_mode = W.Dropdown(options=[("reflect", "reflect"), ("zeros", "zeros"), ("repeat", "repeat")],
                     value="reflect", description="padding:")
_s_sr   = W.Dropdown(options=[4000, 8000, 16000], value=SR_D, description="sample rate:",
                     style={"description_width":"110px"})
_out4   = W.Output()
_s_busy = False   # True while the cycle menu is being refilled

def _refresh4(*_):
    if _s_busy:
        return
    with _out4:
        clear_output(wait=True)
        show_segment(_s_rec.value, _s_cyc.value, _s_dur.value, _s_mode.value, _s_sr.value)

def _on_srec(*_):
    global _s_busy
    _s_busy = True
    _s_cyc.options = cycle_options(_s_rec.value)
    _s_cyc.value = 0
    _s_busy = False
    _refresh4()

_s_rec.observe(_on_srec, "value")
for _w in (_s_cyc, _s_dur, _s_mode, _s_sr):
    _w.observe(_refresh4, "value")
display(W.VBox([W.HBox([_s_rec, _s_cyc]), W.HBox([_s_dur, _s_mode]), _s_sr, _out4]))
_on_srec()

---
## 5 | Mel spectrogram

Turns a 4 s segment (default filter, reflect padding) into a mel spectrogram: a picture of how much
energy each frequency band carries over time. The grey image on the right is what the image models
(ResNet-50, EfficientNetV2-S, Swin-V2-B) were given.

| Setting | What it does |
|---|---|
| **n_fft** | Length of each analysis window. Larger gives finer frequency detail but blurs time; smaller sharpens time, which helps with short crackles. |
| **hop** | Step between windows. Smaller gives more time frames, a wider image. |
| **n_mels** | Number of mel frequency bands, the image height. Fewer bands give a coarser picture. |
| **fmax** | Highest frequency shown. It can't exceed half the sample rate (2000 Hz at 4 kHz). |

The defaults (n_fft 512, hop 128, 128 mels, fmax 2000 Hz) are the settings used for the image models.

**Try:** compare n_fft 256 and 2048 on a crackle cycle, then drop n_mels to 32.

In [ ]:
def mel_image(seg, sr, n_fft=512, hop=128, n_mels=128, fmax=None):
    """Mel spectrogram in dB (power convention, 10*log10). Returns (image, extent)."""
    fmax = float(fmax or sr / 2)
    fmax = min(fmax, sr / 2)
    n_fft = min(int(n_fft), max(len(seg), 64))
    M = librosa.feature.melspectrogram(y=seg, sr=sr, n_fft=n_fft, hop_length=int(hop),
                                       n_mels=int(n_mels), fmin=0.0, fmax=fmax, power=2.0)
    D = librosa.power_to_db(M, ref=np.max)
    return D, [0, len(seg) / sr, 0, fmax]

def show_mel(name, cycle_ix, duration_s, n_fft, hop, n_mels, fmax, target_sr):
    y0, sr0 = load_recording(name)
    y  = librosa.resample(y0, orig_sr=sr0, target_sr=target_sr) if sr0 != target_sr else y0.copy()
    sr = target_sr
    y  = apply_filter(y, sr, "band", LO_D, HI_D, ORD_D)
    cyc = load_cycles(name)
    st, en, k = cyc[min(cycle_ix, len(cyc) - 1)]
    seg, _, pad = cut_cycle(y, sr, st, en, duration_s, "reflect")
    D, ext = mel_image(seg, sr, n_fft, hop, n_mels, fmax)

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2),
                           gridspec_kw={"width_ratios": [1.9, 1]})
    im = ax[0].imshow(D, origin="lower", aspect="auto", extent=ext, cmap="magma")
    ax[0].set_xlabel("Time (s)"); ax[0].set_ylabel("Mel frequency (Hz)")
    ax[0].set_title(f"Cycle {min(cycle_ix, len(cyc) - 1) + 1}  |  {LABELS[k]}  |  "
                    f"{D.shape[0]} mels x {D.shape[1]} frames")
    fig.colorbar(im, ax=ax[0], label="Power (dB re max)")

    sq = np.array(np.clip((D - D.min()) / max(float(np.ptp(D)), 1e-9), 0, 1) * 255, dtype=np.uint8)
    ax[1].imshow(sq, origin="lower", aspect="equal", cmap="gray",
                 extent=[0, 1, 0, 1], interpolation="bilinear")
    ax[1].set_title("what the image models see\n(grey, resized)")
    ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].grid(False)
    plt.show()
    print(f"window {n_fft/sr*1000:.0f} ms, one frame every {hop/sr*1000:.0f} ms")

In [ ]:
# --- interactive ---
_m_rec  = W.Dropdown(options=list(RECS), value=list(RECS)[0], description="Recording:",
                     layout=W.Layout(width="420px"))
_m_cyc  = W.Dropdown(description="Cycle:", layout=W.Layout(width="320px"))
_m_nfft = W.SelectionSlider(options=[128, 256, 512, 1024, 2048], value=512, description="n_fft:",
                            continuous_update=False)
_m_hop  = W.SelectionSlider(options=[32, 64, 128, 256, 512], value=128, description="hop:",
                            continuous_update=False)
_m_mels = W.SelectionSlider(options=[16, 32, 64, 128], value=128, description="n_mels:",
                            continuous_update=False)
_m_fmax = W.FloatSlider(value=2000, min=500, max=2000, step=100, description="fmax (Hz):",
                        continuous_update=False, readout_format=".0f",
                        style={"description_width":"110px"})
_out5   = W.Output()
_m_busy = False   # True while the cycle menu is being refilled

def _refresh5(*_):
    if _m_busy:
        return
    with _out5:
        clear_output(wait=True)
        show_mel(_m_rec.value, _m_cyc.value, 4.0, _m_nfft.value, _m_hop.value,
                 _m_mels.value, _m_fmax.value, SR_D)

def _on_mrec(*_):
    global _m_busy
    _m_busy = True
    _m_cyc.options = cycle_options(_m_rec.value)
    _m_cyc.value = 0
    _m_busy = False
    _refresh5()

_m_rec.observe(_on_mrec, "value")
for _w in (_m_cyc, _m_nfft, _m_hop, _m_mels, _m_fmax):
    _w.observe(_refresh5, "value")
display(W.VBox([W.HBox([_m_rec, _m_cyc]), W.HBox([_m_nfft, _m_hop]),
                W.HBox([_m_mels, _m_fmax]), _out5]))
_on_mrec()

### ✏️ Try it yourself: other time-frequency representations

Implement a different time-frequency representation you know (a wavelet scalogram, the Wigner-Ville
distribution, a constant-Q transform, ...) and compare it with the mel spectrogram on the same cycle.
Can you see crackles and wheezes more clearly in it? You can reuse `cut_cycle(...)` and `mel_image(...)`.

*The cell below is yours to experiment in.*

In [ ]:
# Your code here

---
## 6 | Train a classifier

Each model (backbone) was run once over the whole dataset in advance, and its output vectors
(*embeddings*) were saved. Here you pick a model, load its embeddings and train a small classifier
head on top. Training takes a few seconds; the epoch with the best validation score is kept.

| Setting | What it does |
|---|---|
| **Model** | Whose embeddings to use. ResNet-50, EfficientNetV2-S and Swin-V2-B look at the mel image; VGGish and YAMNet take the audio directly. |
| **Optimizer** | *Adam* adapts the step size for each weight and usually works out of the box; *SGD + momentum* is simpler and more sensitive to the learning rate. |
| **Learning rate** | Size of each update step. Too small learns slowly; too large becomes unstable. |
| **Epochs** | Number of passes over the training data. |
| **Class weights** | Most cycles are Normal, so a head tends to favour Normal. *Balanced* gives rarer classes more weight automatically; *Custom* lets you set each class's weight with the sliders. A higher weight makes mistakes on that class cost more. |

Validation is a random 15% of the training cycles, so its score is optimistic. The test score in
section 7 is the one that counts.

**Try:** train VGGish and YAMNet with the same settings, then retrain the better one with *Balanced*
class weights and compare Se and Sp.

In [ ]:
# --- embeddings -------------------------------------------------------------
# The models offered in this notebook, in menu order.
MODEL_NAMES = {"resnet50": "ResNet-50", "efficientnet_v2_s": "EfficientNetV2-S",
               "swin_v2_b": "Swin-V2-B", "vggish": "VGGish", "yamnet": "YAMNet"}

def model_name(key):
    return MODEL_NAMES.get(key, key)

def model_choices():
    """Dropdown entries (display name, key)."""
    return [(name, k) for k, name in MODEL_NAMES.items() if k in MANIFEST["models"]]

def is_logits(key):
    """True when the saved vectors are ImageNet class scores rather than features (see manifest)."""
    m = MANIFEST["models"][key]
    return m["embedding_dim"] == 1000 and "logit" in (m.get("caveat") or "").lower()

def load_split(model_key):
    """Return dict with Xtr, ytr, Xva, yva, Xte, yte (embeddings float32, labels int64)."""
    E = lambda s: np.load(os.path.join(DATA_DIR, "embeddings", f"{model_key}_{s}.npy")).astype(np.float32)
    L = lambda f: np.load(os.path.join(DATA_DIR, f)).astype(np.int64)
    meta = MANIFEST["models"][model_key]
    Xtr  = E("train")
    ytr  = L(meta["train_labels_file"])        # train row order differs per model
    Xte, yte = E("test"), L("labels_test.npy")
    Xva, yva = E("val"), L("labels_val.npy")
    assert len(Xtr) == len(ytr) and len(Xva) == len(yva) and len(Xte) == len(yte)
    return dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte)

In [ ]:
# --- a small classifier head, in numpy --------------------------------------
# No torch and no GPU needed: the heavy backbones already ran, this part is tiny.

BATCH, HIDDEN = 128, 128

def _onehot(y, k):
    o = np.zeros((len(y), k), np.float32); o[np.arange(len(y)), y] = 1.0; return o

def _softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

def icbhi_scores(y_true, y_pred):
    """Official ICBHI 2017 scores.
    Sp = share of Normal cycles called Normal. Se = share of abnormal cycles given the right class.
    Score = (Se + Sp) / 2.
    """
    norm = y_true == NORMAL
    abn  = ~norm
    sp = float((y_pred[norm] == y_true[norm]).mean()) if norm.any() else float("nan")
    se = float((y_pred[abn]  == y_true[abn]).mean())  if abn.any()  else float("nan")
    return se, sp, (se + sp) / 2.0

def balanced_weights(y, k=None):
    """Inverse-frequency class weights, N / (K * count): rarer classes get larger weights."""
    k = k or len(LABELS)
    cnt = np.bincount(y, minlength=k).astype(np.float32)
    return (len(y) / (k * np.maximum(cnt, 1))).astype(np.float32)

def train_head(data, optimizer="adam", lr=1e-3, class_weights=None,
               epochs=20, hidden=HIDDEN, seed=0, log=print):
    """Train the head; class_weights is None (all equal) or one weight per class."""
    rng = np.random.default_rng(seed)
    Xtr, ytr, Xva, yva = data["Xtr"], data["ytr"], data["Xva"], data["yva"]
    K, d = len(LABELS), Xtr.shape[1]

    mu = Xtr.mean(0); sd = Xtr.std(0); sd[sd < 1e-6] = 1.0        # standardise with TRAIN statistics
    Ztr, Zva = (Xtr - mu) / sd, (Xva - mu) / sd

    W1 = (rng.standard_normal((d, hidden)) * np.sqrt(2.0 / d)).astype(np.float32)
    b1 = np.zeros(hidden, np.float32)
    W2 = (rng.standard_normal((hidden, K)) * np.sqrt(2.0 / hidden)).astype(np.float32)
    b2 = np.zeros(K, np.float32)
    P  = [W1, b1, W2, b2]
    state = [[np.zeros_like(p) for p in P], [np.zeros_like(p) for p in P]]
    cw = np.ones(K, np.float32) if class_weights is None else np.asarray(class_weights, np.float32)

    def fwd(Z):
        H = np.maximum(Z @ P[0] + P[1], 0.0)
        return H, _softmax(H @ P[2] + P[3])

    best, hist = None, []
    with np.errstate(over="ignore", invalid="ignore"):          # a very high learning rate may diverge
        for ep in range(1, epochs + 1):
            order = rng.permutation(len(Ztr))
            for s in range(0, len(order), BATCH):
                ix = order[s:s + BATCH]
                Z, y = Ztr[ix], ytr[ix]
                H, Pr = fwd(Z)
                G  = (Pr - _onehot(y, K)) * cw[y][:, None] / len(ix)   # weighted cross-entropy gradient
                gW2 = H.T @ G;  gb2 = G.sum(0)
                dH  = (G @ P[2].T) * (H > 0)
                gW1 = Z.T @ dH; gb1 = dH.sum(0)
                for i, g in enumerate([gW1, gb1, gW2, gb2]):
                    if optimizer == "sgd":
                        state[0][i] = 0.9 * state[0][i] + g
                        P[i] -= lr * state[0][i]
                    else:  # adam
                        state[0][i] = 0.9  * state[0][i] + 0.1  * g
                        state[1][i] = 0.999 * state[1][i] + 0.001 * (g * g)
                        mh = state[0][i] / (1 - 0.9 ** ep)
                        vh = state[1][i] / (1 - 0.999 ** ep)
                        P[i] -= lr * mh / (np.sqrt(vh) + 1e-8)
            se, sp, sc = icbhi_scores(yva, fwd(Zva)[1].argmax(1))
            hist.append((ep, se, sp, sc))
            if best is None or sc > best["score"]:
                best = {"score": sc, "epoch": ep, "P": [p.copy() for p in P]}
    if not all(np.isfinite(p).all() for p in P):
        log("Warning: training diverged (weights became infinite). Try a lower learning rate.")
    log(f"Best epoch {best['epoch']} of {epochs}, validation ICBHI score {best['score']:.3f}")
    return {"params": best["P"], "mu": mu, "sd": sd, "hidden": hidden,
            "best_epoch": best["epoch"], "val_score": best["score"], "history": hist,
            "optimizer": optimizer, "lr": lr, "epochs": epochs,
            "class_weights": None if class_weights is None else [float(w) for w in cw]}

RUNS   = []     # one row per result (the Train button or your own head), listed in section 7
RESULT = {}     # the latest test result, shown in section 7

def record_result(model_key, head, settings, y_true, y_pred):
    """Store a test result for section 7. Returns (Se, Sp, ICBHI score)."""
    se, sp, sc = icbhi_scores(y_true, y_pred)
    RESULT.clear()
    RESULT.update(model=model_name(model_key), head=head, settings=settings,
                  y_true=y_true, y_pred=y_pred, se=se, sp=sp, score=sc)
    RUNS.append(dict(model=model_name(model_key), head=head, settings=settings,
                     se=se, sp=sp, score=sc))
    return se, sp, sc

def show_my_results(model_key, test_output, name="My head"):
    """Score your own head on the test set and show it in section 7.

    test_output: class probabilities, shape (test cycles, 4), or the predicted class per cycle.
    """
    yte = np.load(os.path.join(DATA_DIR, "labels_test.npy")).astype(np.int64)
    out = np.asarray(test_output)
    y_pred = out.argmax(1) if out.ndim == 2 else out.astype(np.int64)
    if len(y_pred) != len(yte) or (out.ndim == 2 and out.shape[1] != len(LABELS)):
        raise ValueError(f"expected shape ({len(yte)}, {len(LABELS)}) or ({len(yte)},), got {out.shape}")
    se, sp, sc = record_result(model_key, name, "your own head", yte, y_pred)
    print(f"{name} on {model_name(model_key)}:  Se {se:.3f}   Sp {sp:.3f}   ICBHI score {sc:.3f}"
          "   (full results in section 7)")
    if "_show7" in globals():
        _show7()

def check_my_onnx(path, model_key, name=None):
    """Check that your ONNX head works like ours, score it on the test set and download it (Colab)."""
    if not ensure("onnxruntime"):
        print("onnxruntime is not available."); return
    import onnxruntime as ort
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    inp, out = s.get_inputs()[0], s.get_outputs()[0]
    X = load_split(model_key)["Xte"]
    P = s.run([out.name], {inp.name: X})[0]
    if inp.name != "embedding" or out.name != "probabilities":
        print(f"Note: name the input 'embedding' and the output 'probabilities' "
              f"(yours: '{inp.name}', '{out.name}').")
    if P.shape != (len(X), len(LABELS)):
        print(f"Problem: the output must have shape (N, {len(LABELS)}), got {P.shape}."); return
    if not np.allclose(P.sum(1), 1.0, atol=1e-3) or P.min() < 0:
        print("Note: the outputs are not probabilities. Add a softmax at the end of your head.")
    show_my_results(model_key, P, name or os.path.splitext(os.path.basename(path))[0])
    if IN_COLAB:
        from google.colab import files
        files.download(path)

def predict(model, X):
    """Class probabilities for raw embeddings X."""
    W1, b1, W2, b2 = model["params"]
    Z = (X - model["mu"]) / model["sd"]
    with np.errstate(over="ignore", invalid="ignore"):
        H = np.maximum(Z @ W1 + b1, 0.0)
        return _softmax(H @ W2 + b2)

In [ ]:
# --- interactive ---
TRAINED = {}      # the last head trained with the Train button; section 8 exports it

_t_model = W.Dropdown(options=model_choices(), description="Model:")
_t_opt   = W.Dropdown(options=[("Adam", "adam"), ("SGD + momentum", "sgd")],
                      value="adam", description="Optimizer:")
_LRS     = [1e-5, 2e-5, 5e-5, 1e-4, 2e-4, 5e-4, 1e-3, 2e-3, 5e-3, 1e-2, 2e-2, 5e-2, 1e-1, 2e-1, 5e-1]
_t_lr    = W.SelectionSlider(options=[(f"{v:g}", v) for v in _LRS], value=1e-3,
                             description="Learning rate:", continuous_update=False,
                             style={"description_width":"110px"}, layout=W.Layout(width="420px"))
_t_ep    = W.IntSlider(value=20, min=10, max=100, step=5, description="Epochs:",
                       continuous_update=False, style={"description_width":"110px"},
                       layout=W.Layout(width="420px"))
_t_cwm   = W.ToggleButtons(options=["None", "Balanced", "Custom"], value="None",
                           description="Class weights:", style={"description_width":"110px",
                                                                "button_width":"90px"})
_t_cws   = {n: W.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description=f"{n}:",
                             readout_format=".1f", continuous_update=False)
            for n in LABELS}
_t_btn   = W.Button(description="Train", button_style="primary", icon="play")
_out6    = W.Output()

def _sync_weights(*_):
    """None: all 1. Balanced: computed from the model's training labels. Custom: you set them."""
    mode = _t_cwm.value
    if mode == "Balanced":
        meta = MANIFEST["models"][_t_model.value]
        y = np.load(os.path.join(DATA_DIR, meta["train_labels_file"])).astype(np.int64)
        vals = balanced_weights(y)
    for i, n in enumerate(LABELS):
        if mode == "None":
            _t_cws[n].value = 1.0
        elif mode == "Balanced":
            _t_cws[n].value = round(float(vals[i]), 1)
        _t_cws[n].disabled = mode != "Custom"

_t_cwm.observe(_sync_weights, "value")
_t_model.observe(_sync_weights, "value")
_sync_weights()

def _do_train(_=None):
    _t_btn.disabled = True
    try:
        with _out6:
            clear_output(wait=True)
            key, name, epochs = _t_model.value, model_name(_t_model.value), _t_ep.value
            data = load_split(key)
            cw = None if _t_cwm.value == "None" else [_t_cws[n].value for n in LABELS]
            print(f"Training a head on {name} embeddings ({data['Xtr'].shape[1]} values per cycle, "
                  f"{len(data['ytr'])} training cycles) ...")
            if is_logits(key):
                print(f"Note: the {name} vectors are its 1000 ImageNet class scores, not internal features.")
            m = train_head(data, _t_opt.value, _t_lr.value, cw, epochs)
            pte = predict(m, data["Xte"]).argmax(1)
            settings = (f"{_t_opt.label}, learning rate {_t_lr.value:g}, {epochs} epochs "
                        f"(kept epoch {m['best_epoch']}), class weights {_t_cwm.value}")
            se, sp, sc = record_result(key, "built-in", settings, data["yte"], pte)
            m.update(model_key=key, data=data)
            TRAINED.clear(); TRAINED.update(m)
            print(f"Test: Se {se:.3f}   Sp {sp:.3f}   ICBHI score {sc:.3f}   (full results in section 7)")

            ep, s_se, s_sp, s_sc = zip(*m["history"])
            fig, ax = plt.subplots(figsize=(6.4, 2.4))
            ax.plot(ep, s_sc, "-o", ms=3, color="#0b6e75", label="ICBHI score")
            ax.plot(ep, s_se, lw=0.9, color="#c2622a", label="Se")
            ax.plot(ep, s_sp, lw=0.9, color="#9aa5ad", label="Sp")
            ax.axvline(m["best_epoch"], ls="--", lw=0.9, color="#2b3036")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Validation score"); ax.set_ylim(0, 1)
            ax.set_title(f"{name}: validation scores per epoch (dashed line = kept epoch)")
            ax.legend(fontsize=8); plt.show()
    finally:
        _t_btn.disabled = False
    if "_show7" in globals():          # refresh the results view in section 7
        _show7()

_t_btn.on_click(_do_train)
display(W.VBox([W.HBox([_t_model, _t_opt]), _t_lr, _t_ep, _t_cwm,
                W.HBox([_t_cws[n] for n in LABELS[:2]]), W.HBox([_t_cws[n] for n in LABELS[2:]]),
                _t_btn, _out6]))

### ✏️ Try it yourself: build your own head

Replace our head with your own classifier: a deeper network, an SVM, anything you like. Load the
embeddings with `data = load_split("vggish")` (keys `Xtr`, `ytr`, `Xva`, `yva`, `Xte`, `yte`) and
train on them. Then:

- **See its results:** `show_my_results("vggish", test_probabilities)` scores it on the test set and
  shows it in section 7, next to your other runs. `test_probabilities` has shape (test cycles, 4); the
  predicted class per cycle also works.
- **Save it for Auralis (section 9):** export it as ONNX with the same interface as ours. Input: raw
  embeddings of shape `[N, d]` named `embedding` (any input scaling goes inside the model). Output:
  class probabilities of shape `[N, 4]` named `probabilities` (include the softmax), in the order of
  `LABELS`. In PyTorch, which is already in Colab:
  `torch.onnx.export(my_head, torch.zeros(1, d), "my_head.onnx", input_names=["embedding"], output_names=["probabilities"], dynamic_axes={"embedding": {0: "N"}, "probabilities": {0: "N"}}, dynamo=False)`
- **Check the file:** `check_my_onnx("my_head.onnx", "vggish")` checks the interface, scores it on the
  test set and downloads it.

*The cell below is yours to experiment in.*

In [ ]:
# Your code here

---
## 7 | Results

Shows how the latest head does on the **test set**, the official ICBHI test patients, never seen in
training. It updates automatically every time you press **Train** or call `show_my_results` in
section 6.

- **Se** (sensitivity): share of abnormal cycles (Crackle, Wheeze, Both) given exactly the right class.
- **Sp** (specificity): share of Normal cycles called Normal.
- **ICBHI score**: the mean of Se and Sp, the standard number reported for this dataset.
- **Accuracy**: share of all cycles classified correctly. Misleading here, because most cycles are Normal.
- **Macro F1**: the F1 score averaged over the four classes, so rare classes count as much as Normal.
- **Recall per class**: share of each class's cycles that were classified correctly.
- **Confusion matrix**: rows are the true class, columns the predicted class, and the diagonal is
  correct. The numbers are cycle counts; the colour is the share of each row.

A head that always answers Normal scores Se 0, Sp 1 and an ICBHI score of 0.5, so always read Se and
Sp together. The table at the bottom lists all your runs so you can compare models, heads and settings.

In [ ]:
from IPython.display import HTML

def confusion(y_true, y_pred, k):
    C = np.zeros((k, k), int)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
    return C

def macro_f1(C):
    f1 = []
    for i in range(len(C)):
        tp = C[i, i]; fp = C[:, i].sum() - tp; fn = C[i, :].sum() - tp
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(f1)), f1

def show_results():
    if not RESULT:
        print("No results yet. Press Train in section 6 and they will appear here.")
        return
    r = RESULT
    yte, pte = r["y_true"], r["y_pred"]
    C = confusion(yte, pte, len(LABELS))
    acc = float((pte == yte).mean())
    mf1, _ = macro_f1(C)
    recall = C.diagonal() / np.maximum(C.sum(1), 1)

    print(f"{r['model']}  |  {r['head']} head: {r['settings']}  |  test set: {len(yte)} cycles")

    names  = ["Se", "Sp", "ICBHI score", "Accuracy", "Macro F1"] + [f"Recall {n}" for n in LABELS]
    vals   = [r["se"], r["sp"], r["score"], acc, mf1] + list(recall)
    colors = ["#1f4e79"] * 3 + ["#5d6b75"] * 2 + [CLR[n] for n in LABELS]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.9), gridspec_kw={"width_ratios": [1.15, 1]})
    yy = np.arange(len(names))[::-1]
    ax[0].barh(yy, vals, color=colors)
    for y_, v in zip(yy, vals):
        ax[0].text(v + 0.01, y_, f"{v:.3f}", va="center", fontsize=8)
    ax[0].set_yticks(yy, names); ax[0].set_xlim(0, 1.12); ax[0].set_title("Test metrics")

    Cn = C / np.maximum(C.sum(1, keepdims=True), 1)
    im = ax[1].imshow(Cn, cmap="Blues", vmin=0, vmax=1)
    ax[1].set_xticks(range(len(LABELS)), LABELS, rotation=45, ha="right")
    ax[1].set_yticks(range(len(LABELS)), LABELS)
    ax[1].set_xlabel("Predicted class"); ax[1].set_ylabel("True class")
    ax[1].set_title("Confusion matrix")
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax[1].text(j, i, f"{C[i,j]}", ha="center", va="center", fontsize=8,
                       color="white" if Cn[i, j] > 0.5 else "#2b3036")
    ax[1].grid(False); fig.colorbar(im, ax=ax[1], label="share of true class")
    plt.show()

    # all runs so far, best ICBHI score in bold
    best = max(range(len(RUNS)), key=lambda i: RUNS[i]["score"])
    head = "".join(f"<th style='padding:2px 10px'>{h}</th>" for h in
                   ["#", "Model", "Head", "Settings", "Se", "Sp", "ICBHI score"])
    rows = ""
    for i, run in enumerate(RUNS):
        cells = [i + 1, run["model"], run["head"], run["settings"],
                 f"{run['se']:.3f}", f"{run['sp']:.3f}", f"{run['score']:.3f}"]
        style = "font-weight:bold" if i == best else ""
        rows += f"<tr style='{style}'>" + "".join(f"<td style='padding:2px 10px'>{c}</td>"
                                                 for c in cells) + "</tr>"
    display(HTML(f"<b>Your runs</b><table><tr>{head}</tr>{rows}</table>"))

In [ ]:
# --- results view (refreshes after every new result) ---
_out7 = W.Output()

def _show7():
    with _out7:
        clear_output(wait=True)
        show_results()

display(_out7)
_show7()

---
## 8 | Export to ONNX

Saves the last head trained with the **Train** button as an ONNX file, ready to upload to Auralis in
section 9. In Colab it downloads automatically.

The file takes a raw embedding from the **same model** as input (the input scaling is built in) and
returns the probability of each class.

In [ ]:
def export_onnx(out_dir=".", filename=None):
    if not TRAINED:
        print("Nothing trained yet — press Train in section 6 first."); return None
    if not ensure("onnx"):
        print("onnx not available; cannot export."); return None
    import onnx
    from onnx import helper, TensorProto, numpy_helper

    m = TRAINED
    W1, b1, W2, b2 = m["params"]
    mu, sd = m["mu"], m["sd"]
    # fold the input standardisation (x - mu) / sd into layer 1, so the file takes raw embeddings
    W1f = (W1 / sd[:, None]).astype(np.float32)
    b1f = (b1 - (mu / sd) @ W1).astype(np.float32)
    d, K = W1f.shape[0], W2.shape[1]

    init = [numpy_helper.from_array(W1f, "W1"), numpy_helper.from_array(b1f, "b1"),
            numpy_helper.from_array(W2.astype(np.float32), "W2"),
            numpy_helper.from_array(b2.astype(np.float32), "b2")]
    nodes = [helper.make_node("Gemm", ["embedding", "W1", "b1"], ["h_pre"]),
             helper.make_node("Relu", ["h_pre"], ["h"]),
             helper.make_node("Gemm", ["h", "W2", "b2"], ["logits"]),
             helper.make_node("Softmax", ["logits"], ["probabilities"], axis=1)]
    graph = helper.make_graph(
        nodes, f"bsn_head_{m['model_key']}",
        [helper.make_tensor_value_info("embedding", TensorProto.FLOAT, ["N", d])],
        [helper.make_tensor_value_info("probabilities", TensorProto.FLOAT, ["N", K])],
        initializer=init)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)],
                              producer_name="bsn-workshop")
    model.ir_version = 9
    onnx.checker.check_model(model)

    onnx_path = os.path.join(out_dir, f"{filename or 'bsn_head_' + m['model_key']}.onnx")
    onnx.save(model, onnx_path)
    print(f"{onnx_path}   {os.path.getsize(onnx_path)/1024:.0f} KB   "
          f"({model_name(m['model_key'])} embeddings in, {K} class probabilities out)")

    if ensure("onnxruntime"):
        import onnxruntime as ort
        s = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        X = m["data"]["Xte"][:64].astype(np.float32)
        got = s.run(None, {"embedding": X})[0]
        ref = predict(m, X)
        print(f"onnxruntime check: max|onnx - numpy| = {np.abs(got - ref).max():.2e}")
    else:
        print("onnxruntime unavailable — graph written but not numerically verified.")

    if IN_COLAB:
        from google.colab import files
        files.download(onnx_path)
    return onnx_path

In [ ]:
# --- interactive ---
_e_btn = W.Button(description="Export ONNX", button_style="success", icon="download")
_out8  = W.Output()

def _do_export(_=None):
    with _out8:
        clear_output(wait=True)
        export_onnx(".")

_e_btn.on_click(_do_export)
display(W.VBox([_e_btn, _out8]))

---
## 9 | Explore your model on Auralis

Upload the ONNX file from section 8, or your own head, to **[Auralis](https://myauralis.pt/)** and
pick the backbone you trained it on. There you can explore the data and see why your model decides
the way it does, with example-based explanations (XAI). Try heads from different backbones and
compare them.

---

**Dataset.** Rocha, B. M. et al. (2019). *An open access database for the evaluation of respiratory
sound classification algorithms.* Physiological Measurement, 40(3), 035001.